# Dino Slayer — Notebook 1: Data Preparation

**ASEAN GeoAI Fusion 2026 · Consumer Empowerment · Area of interest: Sabah, Malaysia**

This notebook covers the first three stages of the geospatial pipeline taught in
**Module 6, Session 2 (Data Fusion)**:

| Stage | Meaning | Where in this notebook |
|---|---|---|
| **1. Raw data** | unprocessed observations, no context or structure | D1–D8 downloads |
| **2. Information** | cleaned and georeferenced into structured datasets | clipping to Sabah, CRS handling |
| **3. Features and indicators** | patterns and AI-ready inputs derived from the information | P2 feature join, DIPI v0 |

Stages 4 (predictions) and 5 (actionable decisions) are in notebooks 02 and 03.

---

### What data fusion means here

No single source describes a community's digital inclusion. Ookla measures observed
mobile performance but only where people ran tests. OpenStreetMap maps settlements and
facilities but with uneven rural completeness. WorldPop models population but does not
observe it. **Fusion at the settlement level** combines eight sources into one structured
table, so that each of Sabah's 1,448 mapped settlements carries the evidence from all of them.

The bootcamp calls the output an *AI-ready input*. We call the product a
**settlement-level multi-layer feature stack**, and not a data cube, because the layers are
joined to points rather than stacked on a common raster grid.

---

### Datasets fused

| ID | Source | Licence / credit |
|---|---|---|
| D1 | GADM 4.1 administrative boundaries | GADM |
| D2 | Ookla Speedtest Open Data, mobile tiles 2025 Q1–Q4 | CC BY-NC-SA 4.0 |
| D3/D5 | OpenStreetMap settlements + facilities | ODbL, © OpenStreetMap contributors |
| D4 | WorldPop 2020 constrained population | CC BY 4.0 |
| D6 | JRC Global Surface Water seasonality 2021 | © EC JRC / Google |
| D7 | Meta Relative Wealth Index | CC BY 4.0 |
| D8 | NASA SRTM elevation (30 m) | public domain |
| — | OpenCelliD cell records (context layer only) | CC BY-SA 4.0 |

---

### Before running

Place these in `MyDrive/Dataset/`:

- `502.csv.gz` — OpenCelliD Malaysia export
  https://drive.google.com/file/d/1Tv0JZ-o1bGFHtAeOTk9OByqrFpHJ5CG2/view?usp=sharing

Everything else is downloaded from source by this notebook.

## 📚 Data Sources and Citations

All datasets used in this notebook are open and freely available. Sources, licences and
required attribution are listed below.

| ID | Dataset | Source | Licence | Required credit |
|---|---|---|---|---|
| **D1** | GADM 4.1 administrative boundaries | [gadm.org](https://gadm.org/download_country.html) | Free for academic / non-commercial use | Boundaries: GADM |
| **D2** | Ookla Speedtest Open Data (mobile tiles, 2025 Q1–Q4) | [registry.opendata.aws](https://registry.opendata.aws/speedtest-global-performance/) | CC BY-NC-SA 4.0 | Network performance data © Ookla, Speedtest Open Data |
| **D3** | OpenStreetMap — settlements | [Geofabrik](https://download.geofabrik.de/asia/malaysia-singapore-brunei.html) | ODbL | © OpenStreetMap contributors |
| **D4** | WorldPop 2020 constrained, UN-adjusted | [hub.worldpop.org](https://hub.worldpop.org/geodata/listing?id=29) | CC BY 4.0 | Population data © WorldPop |
| **D5** | OpenStreetMap — schools & clinics | [Geofabrik](https://download.geofabrik.de/asia/malaysia-singapore-brunei.html) | ODbL | © OpenStreetMap contributors |
| **D6** | JRC Global Surface Water (seasonality, 2021) | [global-surface-water.appspot.com](https://global-surface-water.appspot.com/download) | Free for all use | © EC JRC / Google |
| **D7** | Meta Relative Wealth Index | [data.humdata.org](https://data.humdata.org/dataset/relative-wealth-index) | CC BY 4.0 | Relative Wealth Index © Meta Data for Good |
| **D8** | NASA SRTM elevation (30 m) | [earthdata.nasa.gov](https://www.earthdata.nasa.gov/sensors/srtm) | Public domain | Elevation data: NASA SRTM |
| — | OpenCelliD (context layer only) | [opencellid.org](https://opencellid.org/) | CC BY-SA 4.0 | Cell records © OpenCelliD |

### Exact download endpoints used

| Dataset | URL |
|---|---|
| GADM L1 / L2 | `https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_MYS_{1,2}.json.zip` |
| Ookla | `https://ookla-open-data.s3.amazonaws.com/parquet/performance/type=mobile/year=2025/quarter={1-4}/…` |
| OpenStreetMap | `https://download.geofabrik.de/asia/malaysia-singapore-brunei-latest.osm.pbf` |
| WorldPop | `https://data.worldpop.org/GIS/Population/Global_2000_2020_Constrained/2020/BSGM/MYS/mys_ppp_2020_UNadj_constrained.tif` |
| JRC | `https://storage.googleapis.com/global-surface-water/downloads2021/seasonality/seasonality_110E_10Nv1_4_2021.tif` |
| Meta RWI | `https://data.humdata.org/dataset/.../mys_relative_wealth_index.csv` |
| SRTM | via the `elevation` Python package → AWS Terrain Tiles |

### Academic citations

**JRC Global Surface Water**
Pekel, J.-F., Cottam, A., Gorelick, N., Belward, A.S. (2016). *High-resolution mapping of
global surface water and its long-term changes.* **Nature** 540, 418–422.

**NASA SRTM**
Farr, T.G. et al. (2007). *The Shuttle Radar Topography Mission.* **Reviews of Geophysics**
45, RG2004.

### Combined credits line

> Network performance © Ookla Speedtest Open Data (CC BY-NC-SA 4.0) · © OpenStreetMap
> contributors (ODbL) · Population © WorldPop (CC BY 4.0) · Surface water © EC JRC/Google ·
> Relative Wealth Index © Meta Data for Good (CC BY 4.0) · Elevation: NASA SRTM ·
> Boundaries: GADM · Cell records © OpenCelliD (CC BY-SA 4.0)

In [2]:
# ══════════════════════════════════════════════════════════════════════
# SETUP — mount Google Drive and fix one path constant for the whole
# notebook. Every output is written to Drive, never left on the Colab
# runtime, because /content is wiped when the session ends.
# ══════════════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

import os
D = "/content/drive/MyDrive/Dataset/"      # single source of truth for paths
os.makedirs(D, exist_ok=True)              # works on an empty Drive

print("working folder:", D)
print("existing files:")
for f in sorted(os.listdir(D)):
    print("  ", f)

Mounted at /content/drive
working folder: /content/drive/MyDrive/Dataset/
existing files:



---
## D1 — Administrative boundaries (GADM 4.1)

**Stage 1 → 2: raw data becomes information.**

Boundaries are the reference frame for everything else. Level 1 gives the 13 Malaysian
states, which we use to clip every other dataset down to Sabah. Level 2 gives districts,
which later become the **groups for spatial cross-validation** in notebook 02 and the
filter in the dashboard.

Both files are copied into Drive so later cells never depend on the Colab runtime.

In [3]:
!pip install geopandas -q
import geopandas as gpd
print("geopandas", gpd.__version__)

geopandas 1.1.4


In [4]:
# ══════════════════════════════════════════════════════════════════════
# D1 — download GADM level 1 (states) and level 2 (districts).
# GADM 4.1 is a versioned, frozen release, so this is reproducible.
# ══════════════════════════════════════════════════════════════════════
!wget -q https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_MYS_1.json.zip
!unzip -o -q gadm41_MYS_1.json.zip
!wget -q https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_MYS_2.json.zip
!unzip -o -q gadm41_MYS_2.json.zip

states    = gpd.read_file("gadm41_MYS_1.json")
districts = gpd.read_file("gadm41_MYS_2.json")
print(f"{len(states)} states:", states.NAME_1.tolist())
print(f"{len(districts)} districts nationally")

# Copy into Drive — later cells read from Drive, not from /content
import shutil
for f in ["gadm41_MYS_1.json", "gadm41_MYS_2.json"]:
    shutil.copy(f"/content/{f}", D + f)
print("\ncopied to Drive ✓")

# The area of interest.
# CRS is EPSG:4326 (WGS84) for storage and display;
# distance work later reprojects to a projected CRS (EPSG:32650).
sabah = states[states.NAME_1 == "Sabah"].to_crs(4326)
print("Sabah boundary ready ✓  CRS:", sabah.crs)

16 states: ['Johor', 'Kedah', 'Kelantan', 'KualaLumpur', 'Labuan', 'Melaka', 'NegeriSembilan', 'Pahang', 'Perak', 'Perlis', 'PulauPinang', 'Putrajaya', 'Sabah', 'Sarawak', 'Selangor', 'Trengganu']
144 districts nationally

copied to Drive ✓
Sabah boundary ready ✓  CRS: EPSG:4326


---
## D2 — Network performance (Ookla Speedtest Open Data)

**This is the observed evidence the whole project rests on.**

Ookla publishes quarterly aggregates on a global tile grid: average download, upload and
latency per tile, plus the number of tests and devices behind each average.

Two things must be said clearly because they constrain every claim downstream:

1. **It is crowdsourced observed performance, not coverage.** A tile has a value because
   somebody ran a speed test there. Absence of a tile means nobody tested, *not* that
   service is poor.
2. **Test and device counts matter.** A tile backed by 3 tests is not evidence of the same
   weight as one backed by 300. That is why `tests` is carried through the whole pipeline
   and becomes the evidence tier in the P2 join.

### Choosing the area of interest

Sabah was not picked by intuition. The cell below is the **data bake-off**: one global
quarter is downloaded and four candidate states are compared on how much slow-tile evidence
each contains. Sabah had the most tiles and the most slow tiles, so it offered the strongest
evidence base for a digital-inclusion study.

In [5]:
# ══════════════════════════════════════════════════════════════════════
# D2a — AREA OF INTEREST BAKE-OFF
# Downloads one global quarter (~3.3 million tiles) and compares four
# candidate states. This is the evidence behind choosing Sabah.
#
# The `tile` column holds geometry as WKT text, so it must be converted
# into real geometry before any spatial operation.
# ══════════════════════════════════════════════════════════════════════
import pandas as pd

!wget -q "https://ookla-open-data.s3.amazonaws.com/parquet/performance/type=mobile/year=2025/quarter=4/2025-10-01_performance_mobile_tiles.parquet" -O ookla.parquet

df = pd.read_parquet("ookla.parquet")
print(f"{len(df):,} global tiles downloaded")

tiles = gpd.GeoDataFrame(df, geometry=gpd.GeoSeries.from_wkt(df["tile"]), crs=4326)
print("WKT text converted to vector geometry ✓\n")

for s in ["Kelantan", "Sabah", "Pahang", "Trengganu"]:
    aoi    = states[states.NAME_1 == s].to_crs(4326)
    inside = gpd.sjoin(tiles, aoi, predicate="within")   # spatial join: tile within state
    ok     = inside[inside.tests >= 5]                   # ignore tiles with almost no evidence
    slow   = (ok.avg_d_kbps < 25000).sum()               # under 25 Mbps
    p10    = round(ok.avg_d_kbps.quantile(0.10) / 1000, 1)
    print(f"{s:12s} | tiles: {len(inside):5d} | slow (<25 Mbps): {slow:4d} | worst-10%: {p10} Mbps")

print("\n→ Sabah has the most tiles and the most slow tiles. AOI = Sabah (locked).")
del df, tiles

3,311,092 global tiles downloaded
WKT text converted to vector geometry ✓

Kelantan     | tiles:  3275 | slow (<25 Mbps):  179 | worst-10%: 20.5 Mbps
Sabah        | tiles:  5499 | slow (<25 Mbps):  252 | worst-10%: 22.2 Mbps
Pahang       | tiles:  3945 | slow (<25 Mbps):  151 | worst-10%: 26.2 Mbps
Trengganu    | tiles:  2336 | slow (<25 Mbps):   92 | worst-10%: 27.1 Mbps

→ Sabah has the most tiles and the most slow tiles. AOI = Sabah (locked).


In [6]:
# ══════════════════════════════════════════════════════════════════════
# D2b — download the four 2025 quarters and clip each to Sabah.
#
# Each global file is millions of rows; clipping first keeps only ~5,000
# Sabah tiles per quarter. The result is written STRAIGHT TO DRIVE so it
# survives a runtime restart, and `del` frees memory between quarters
# (free Colab will terminate the session otherwise).
# ══════════════════════════════════════════════════════════════════════
from tqdm.notebook import tqdm

urls = {
    "2025q1": "year=2025/quarter=1/2025-01-01_performance_mobile_tiles.parquet",
    "2025q2": "year=2025/quarter=2/2025-04-01_performance_mobile_tiles.parquet",
    "2025q3": "year=2025/quarter=3/2025-07-01_performance_mobile_tiles.parquet",
    "2025q4": "year=2025/quarter=4/2025-10-01_performance_mobile_tiles.parquet",
}

for name, path in tqdm(urls.items(), desc="Ookla quarters"):
    !wget -q "https://ookla-open-data.s3.amazonaws.com/parquet/performance/type=mobile/{path}" -O temp.parquet
    df      = pd.read_parquet("temp.parquet")
    g       = gpd.GeoDataFrame(df, geometry=gpd.GeoSeries.from_wkt(df["tile"]), crs=4326)
    clipped = gpd.sjoin(g, sabah, predicate="within").drop(columns="index_right")
    clipped["quarter"] = name                       # keep the time label on every row
    clipped.to_parquet(D + f"tiles_sabah_{name}.parquet")
    print(f"  {name} → {len(clipped)} Sabah tiles ✓")
    del df, g, clipped                              # free memory before the next quarter

print("\nD2 complete — 4 quarterly files in Drive")

Ookla quarters:   0%|          | 0/4 [00:00<?, ?it/s]

  2025q1 → 4971 Sabah tiles ✓
  2025q2 → 5071 Sabah tiles ✓
  2025q3 → 5060 Sabah tiles ✓
  2025q4 → 5499 Sabah tiles ✓

D2 complete — 4 quarterly files in Drive


---
## D3 / D5 — Settlements and facilities (OpenStreetMap)

**These define the unit of analysis.** Every row in the final table is one OSM `place`
node: a city, town, village or hamlet.

D5 (schools and clinics) comes from the same extract, so both are done together.

Method: download the Malaysia–Singapore–Brunei extract, shrink it to a bounding box around
Sabah with `osmium` (the full file is too large to parse comfortably), then read it with
`pyrosm` and clip precisely to the Sabah boundary with a **spatial join**.

**Known limitation to carry forward:** OSM is contributor-mapped, so rural completeness is
uneven and tagging is inconsistent. A facility count means *mapped* facilities, not all
facilities. This is stated wherever the counts are used.

In [7]:
# ══════════════════════════════════════════════════════════════════════
# D3 — download the OSM extract for Malaysia/Singapore/Brunei
# ══════════════════════════════════════════════════════════════════════
!pip install pyrosm -q
from pyrosm import OSM

!wget -q https://download.geofabrik.de/asia/malaysia-singapore-brunei-latest.osm.pbf -O malaysia.osm.pbf
!ls -lh malaysia.osm.pbf
print("downloaded ✓")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 kB 3.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 79.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.35.1 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.35.1 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.35.1 which is incom

In [8]:
# ══════════════════════════════════════════════════════════════════════
# Shrink to a bounding box around Sabah before parsing.
# Box: lon 115.0–119.7, lat 4.0–7.6. This is a rough rectangle; the exact
# clip to the state boundary happens in the next cell via spatial join.
# ══════════════════════════════════════════════════════════════════════
!apt-get install -y osmium-tool -q
!osmium extract -b 115.0,4.0,119.7,7.6 malaysia.osm.pbf -o sabah.osm.pbf --overwrite
!ls -lh sabah.osm.pbf
print("shrunk ✓")

Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  libboost-program-options1.74.0
The following NEW packages will be installed:
  libboost-program-options1.74.0 osmium-tool
0 upgraded, 2 newly installed, 0 to remove and 3 not upgraded.
Need to get 882 kB of archives.
After this operation, 3,863 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libboost-program-options1.74.0 amd64 1.74.0-14ubuntu3 [311 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 osmium-tool amd64 1.14.0-1 [571 kB]
Fetched 882 kB in 3s (341 kB/s)
Selecting previously unselected package libboost-program-options1.74.0:amd64.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../libboost-program-options1.74.0_1.74.0-14ubuntu3_amd64.deb ...
Unpacking libboost-program-options1.74.0:amd64 (1.74.0-14ubuntu3) ...
Selecting previously u

In [9]:
# ══════════════════════════════════════════════════════════════════════
# D3 — extract settlements.
# keep_nodes=True only: a place is a point, not an area, so ways and
# relations are excluded to avoid duplicate representations.
# ══════════════════════════════════════════════════════════════════════
osm = OSM("sabah.osm.pbf")

places = osm.get_data_by_custom_criteria(
    custom_filter={"place": ["city", "town", "village", "hamlet"]},
    filter_type="keep",
    keep_nodes=True, keep_ways=False, keep_relations=False)
places = places.set_crs(4326)

# Spatial join clips the bounding box down to the real state boundary
places_sabah = gpd.sjoin(places, sabah, predicate="within").drop(columns="index_right")

print(len(places_sabah), "settlements in Sabah ✓")
print(places_sabah["place"].value_counts().to_string())

1448 settlements in Sabah ✓
place
village    1136
hamlet      253
town         54
city          5


In [10]:
# ══════════════════════════════════════════════════════════════════════
# D5 — extract facilities (schools, clinics, hospitals, doctors).
#
# Some facilities are mapped as areas rather than points, so `.centroid`
# reduces them to a single point. The CRS warning this raises is expected:
# centroids of small polygons in degrees are accurate enough at this scale,
# and the join that matters (3 km counts, in notebook P2) is done in metres.
# ══════════════════════════════════════════════════════════════════════
pois = osm.get_pois(custom_filter={"amenity": ["school", "clinic", "hospital", "doctors"]})
pois = pois.set_crs(4326)
pois["geometry"] = pois.geometry.centroid
pois_sabah = gpd.sjoin(pois, sabah, predicate="within").drop(columns="index_right")

print(len(pois_sabah), "facilities in Sabah ✓")
print(pois_sabah["amenity"].value_counts().to_string())

606 facilities in Sabah ✓
amenity
school      460
clinic       83
hospital     46
doctors      17


/tmp/ipykernel_667/345053268.py:11: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["geometry"] = pois.geometry.centroid


In [11]:
# ══════════════════════════════════════════════════════════════════════
# Pull the settlement name out of the OSM `tags` blob, then save both
# layers keeping only the columns we need.
#
# Not every settlement is named. Unnamed rows are KEPT — they are real
# places — which is why every downstream join uses settlement_id, never name.
# ══════════════════════════════════════════════════════════════════════
import json

def get_name(t):
    """OSM tags arrive as a JSON string or dict; return the name if present."""
    if t is None: return None
    if isinstance(t, str):
        try: t = json.loads(t)
        except Exception: return None
    return t.get("name") if isinstance(t, dict) else None

places_sabah["name"] = places_sabah["tags"].apply(get_name)
print("settlements with names:", places_sabah["name"].notna().sum(), "/", len(places_sabah))

def keep_cols(df, wanted):
    return df[[c for c in wanted if c in df.columns] + ["geometry"]]

keep_cols(places_sabah, ["name", "place"]).to_parquet(D + "settlements_sabah.parquet")
keep_cols(pois_sabah,   ["name", "amenity"]).to_parquet(D + "facilities_sabah.parquet")
print("saved ✓ D3 + D5 complete")

settlements with names: 1410 / 1448
saved ✓ D3 + D5 complete


### 📸 Visual check — settlement distribution

Run the cell below to **display a map here.**

What to look for: the points should trace Sabah's populated west coast, the Kinabatangan
corridor and the eastern towns, with the mountainous interior sparse. If points appear over
open sea or outside the state, the spatial join failed.

In [12]:
# Visual verification. GIS practitioners repeatedly flag silent join and
# geometry failures as the biggest practical risk, so every join in this
# project is checked on a map, not just by row count.
import folium
m = folium.Map(location=[5.4, 116.5], zoom_start=7)
for _, r in places_sabah.sample(min(300, len(places_sabah)), random_state=42).iterrows():
    folium.CircleMarker([r.geometry.y, r.geometry.x], radius=2, color="blue").add_to(m)
m

---
## D4 — Population (WorldPop)

**Raster → vector.** WorldPop publishes population as a raster grid. Settlements are points.
The bridge is a **zonal statistic**: buffer each settlement, then sum the raster cells inside
that buffer.

### The degrees-versus-metres trap

A buffer of "2000" in EPSG:4326 means 2000 *degrees*, which is meaningless. The data must be
reprojected to a **projected CRS in metres** before buffering — EPSG:32650 (UTM zone 50N)
covers Sabah — and then converted back to EPSG:4326 to match the raster. This is exactly the
reprojection step from **Module 3, Session 7 (Hands-on Lab 2)**, and getting it wrong is a
silent failure: the code runs and the numbers are nonsense.

⚠️ **`pop_2km` values overlap between neighbouring settlements and must never be summed.**
Two villages 1 km apart share most of their 2 km buffer. A total across settlements
multiply-counts the same people.

In [13]:
# ══════════════════════════════════════════════════════════════════════
# D4 — WorldPop 2020 constrained, UN-adjusted, ~100 m resolution.
# (The `maxar_v1` variant of this URL returns an empty file; the BSGM
# UN-adjusted build below is the one that works.)
# ══════════════════════════════════════════════════════════════════════
!pip install rasterstats -q
!wget -q https://data.worldpop.org/GIS/Population/Global_2000_2020_Constrained/2020/BSGM/MYS/mys_ppp_2020_UNadj_constrained.tif -O worldpop_mys.tif
!ls -lh worldpop_mys.tif

import rasterio
r = rasterio.open("worldpop_mys.tif")
print("raster CRS:", r.crs, "| shape:", r.shape)

-rw-r--r-- 1 root root 9.4M Sep 20  2020 worldpop_mys.tif
raster CRS: EPSG:4326 | shape: (7831, 23556)


In [14]:
# ══════════════════════════════════════════════════════════════════════
# Count people within 2 km of each settlement.
#
# THE CRITICAL LINE is the reprojection chain:
#   .to_crs(32650)  → metres, so buffer(2000) means 2000 metres
#   .buffer(2000)   → the 2 km zone
#   .to_crs(4326)   → back to the raster's CRS for the zonal statistic
# ══════════════════════════════════════════════════════════════════════
from rasterstats import zonal_stats

settlements = gpd.read_parquet(D + "settlements_sabah.parquet")
print(len(settlements), "settlements loaded")

buffered = settlements.to_crs(32650).buffer(2000).to_crs(4326)

stats = zonal_stats(buffered, "worldpop_mys.tif", stats="sum", nodata=-99999)
settlements["pop_2km"] = [round(s["sum"]) if s["sum"] else 0 for s in stats]

print("\ndone ✓")
print("⚠ the total below is a DIAGNOSTIC ONLY — buffers overlap, so it is")
print("  not the unique population of Sabah and must never be quoted as such.")
print("diagnostic sum:", settlements["pop_2km"].sum())
print("\n10 largest by pop_2km:")
print(settlements.nlargest(10, "pop_2km")[["name", "place", "pop_2km"]].to_string(index=False))

settlements.to_parquet(D + "settlements_sabah_pop.parquet")
print("\nsaved ✓ D4 complete")

1448 settlements loaded

done ✓
⚠ the total below is a DIAGNOSTIC ONLY — buffers overlap, so it is
  not the unique population of Sabah and must never be quoted as such.
diagnostic sum: 5484733

10 largest by pop_2km:
                      name   place  pop_2km
                    Lintas village    74688
                    Luyang village    70890
                  Sembulan village    68654
                      Lido village    67718
                     Damai village    67207
                 Kolombong village    65181
               Karamunsing village    62928
     Kampung Sembulan Lama village    61286
Kampung Setinggan Sembulan village    59310
             Kampung Botol  hamlet    57596

saved ✓ D4 complete


---
## D6 — Surface water context (JRC Global Surface Water)

The JRC seasonality layer maps how many months of the year each 30 m cell held water,
derived from Landsat imagery 1984–2021.

**This is historic surface-water context, not a flood forecast.** Two limits define how it
may be described:

- It is a **historical observation**, so it says nothing about future flood probability.
- It **includes coastal and tidal water**, so a beach village is flagged for reasons that
  have nothing to do with flood risk.

Permanent water (12 months a year — rivers, lakes, sea) is excluded; only *seasonal* water
of 1–11 months is counted. The resulting flag is called **flood-prone context** and is
deliberately kept out of the DIPI score for exactly these reasons.

In [15]:
# ══════════════════════════════════════════════════════════════════════
# D6 — JRC seasonality tile covering Sabah, then a 1 km zonal statistic.
# 1 km rather than 2 km: adjacency to water is a local property.
# ══════════════════════════════════════════════════════════════════════
!wget -q https://storage.googleapis.com/global-surface-water/downloads2021/seasonality/seasonality_110E_10Nv1_4_2021.tif -O jrc_seasonality.tif
!ls -lh jrc_seasonality.tif

import numpy as np
settlements = gpd.read_parquet(D + "settlements_sabah_pop.parquet")
buffered    = settlements.to_crs(32650).buffer(1000).to_crs(4326)   # metres first, again

def count_seasonal(x):
    """Cells holding water 1-11 months a year. 12 = permanent, excluded."""
    x = x.compressed() if hasattr(x, "compressed") else x[~np.isnan(x)]
    return int(((x >= 1) & (x <= 11)).sum())

stats = zonal_stats(buffered, "jrc_seasonality.tif",
                    add_stats={"seasonal": count_seasonal}, nodata=0)
settlements["seasonal_water_px"] = [s["seasonal"] for s in stats]
settlements["flood_prone"]       = settlements["seasonal_water_px"] >= 5

print("flood-prone context flagged:", settlements["flood_prone"].sum(), "/", len(settlements))
print("\n⚠ 'flood-prone context' = seasonal water adjacency. NEVER 'flood prediction'.")
print("  Includes coastal and tidal water.\n")
print(settlements[settlements.flood_prone].nlargest(8, "seasonal_water_px")[
      ["name","place","pop_2km","seasonal_water_px"]].to_string(index=False))

settlements.to_parquet(D + "settlements_sabah_full.parquet")
print("\nsaved ✓ D6 complete → settlements_sabah_full.parquet (the master file)")

-rw-r--r-- 1 root root 16M Sep 29  2022 jrc_seasonality.tif
flood-prone context flagged: 577 / 1448

⚠ 'flood-prone context' = seasonal water adjacency. NEVER 'flood prediction'.
  Includes coastal and tidal water.

                  name   place  pop_2km  seasonal_water_px
              Semporna    town    41050                513
Kampung Pompod Kelawat village     1550                314
     Kampung Lumanggas  hamlet      233                278
          Kg. Tinusa 2 village    17790                268
        Skim Bumburing village     2486                263
        Kampung Forest  hamlet    41122                256
        Api-Api Centre village    46058                243
         Kota Kinabalu    city    43163                241

saved ✓ D6 complete → settlements_sabah_full.parquet (the master file)


---
## D7 — Relative Wealth Index (Meta Data for Good)

RWI is a modelled estimate of relative wealth on a ~2.4 km grid. Negative is poorer than the
national average, positive is richer.

This becomes the **equity pillar** of DIPI: a community that is both underserved and poor is
less able to remedy the problem privately, so it ranks higher for public attention.

⚠️ **Missing RWI means "no wealth data", not "poor".** Coverage stops in the remotest areas,
which are exactly the places where a wrong assumption would do most damage. Missing values
stay missing through the whole pipeline and are treated as neutral in DIPI.

In [16]:
# ══════════════════════════════════════════════════════════════════════
# D7 — Meta RWI: a CSV of lat/lon points, clipped to Sabah, then averaged
# within 2.4 km of each settlement (2.4 km ≈ the native RWI grid cell).
# ══════════════════════════════════════════════════════════════════════
!wget -q "https://data.humdata.org/dataset/76f2a2ea-ba50-40f5-b79c-db95d668b843/resource/4249dc0a-46a4-4f66-93c6-0d87dec072a6/download/mys_relative_wealth_index.csv" -O rwi_mys.csv

rwi   = pd.read_csv("rwi_mys.csv")
print(len(rwi), "RWI points nationally |", rwi.columns.tolist())

rwi_g     = gpd.GeoDataFrame(rwi, geometry=gpd.points_from_xy(rwi.longitude, rwi.latitude), crs=4326)
rwi_sabah = gpd.sjoin(rwi_g, sabah, predicate="within").drop(columns="index_right")
print(len(rwi_sabah), "RWI points in Sabah ✓")

# Buffer in metres, join, average
settlements = gpd.read_parquet(D + "settlements_sabah_full.parquet")
s_m = settlements.to_crs(32650).copy()
s_m["geometry"] = s_m.buffer(2400)
r_m = rwi_sabah.to_crs(32650)

joined   = gpd.sjoin(r_m, s_m[["name","geometry"]].reset_index(), predicate="within")
mean_rwi = joined.groupby("index")["rwi"].mean()
settlements["rwi"] = settlements.index.map(mean_rwi)

print("\nsettlements with RWI:", settlements["rwi"].notna().sum(), "/", len(settlements))
print("⚠ the rest = 'no wealth data', NOT 'poor'\n")
print("poorest 5:"); print(settlements.nsmallest(5,"rwi")[["name","place","rwi"]].to_string(index=False))
print("\nrichest 3:"); print(settlements.nlargest(3,"rwi")[["name","place","rwi"]].to_string(index=False))

settlements.to_parquet(D + "settlements_sabah_full.parquet")
print("\nsaved ✓ D7 complete — equity pillar loaded")

18147 RWI points nationally | ['latitude', 'longitude', 'rwi', 'error']
4397 RWI points in Sabah ✓

settlements with RWI: 1352 / 1448
⚠ the rest = 'no wealth data', NOT 'poor'

poorest 5:
                name   place     rwi
Kampung Tatalaan Ulu  hamlet -1.1560
                None  hamlet -1.1220
         Malaing Ulu village -1.1220
       Karamatoi Ulu village -1.1220
             Genting  hamlet -1.1205

richest 3:
                 name   place   rwi
             Sandakan    city 1.337
Kampung Berhala Darat village 1.337
  Kampung Air Sim Sim village 1.308

saved ✓ D7 complete — equity pillar loaded


---
## D8 — Elevation (NASA SRTM, 30 m)

Elevation is sampled at each settlement point. It is **not used in DIPI v0** — it is carried
for the model ablations in notebook 02, where terrain is tested as a predictor and, as it
turns out, rejected.

⚠️ **`gdal-bin` must be installed before `eio clip` runs.** The `elevation` package shells
out to `gdalbuildvrt`, which is not on Colab's PATH by default. Without it, `eio` downloads
every tile, fails silently at the merge step, and writes a **0-byte GeoTIFF** — after which
every elevation reading is null with no error raised. This is the single most dangerous
silent failure in the pipeline.

In [17]:
# ══════════════════════════════════════════════════════════════════════
# D8 — SRTM elevation.
# gdal-bin FIRST. See the warning above.
# Sabah is downloaded as four quadrants because eio handles smaller
# bounding boxes more reliably, then merged into one raster.
# ══════════════════════════════════════════════════════════════════════
!apt-get install -y gdal-bin -q
!pip install elevation -q
!which gdalbuildvrt gdal_merge.py          # both must resolve, or the merge fails silently

for name, w, s, e, n in [("a", 115.0, 4.0,  117.35, 5.8),
                         ("b", 117.35, 4.0, 119.7,  5.8),
                         ("c", 115.0, 5.8,  117.35, 7.6),
                         ("d", 117.35, 5.8, 119.7,  7.6)]:
    print("chunk", name, "...")
    !eio clip -o srtm_{name}.tif --bounds {w} {s} {e} {n}

!gdal_merge.py -o srtm_sabah.tif srtm_a.tif srtm_b.tif srtm_c.tif srtm_d.tif
!ls -lh srtm_sabah.tif      # ~419 MB. If this reads 0, gdal-bin did not install.

Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  python3-gdal python3-numpy
Suggested packages:
  libgdal-grass python-numpy-doc python3-dev python3-pytest
The following NEW packages will be installed:
  gdal-bin python3-gdal python3-numpy
0 upgraded, 3 newly installed, 0 to remove and 3 not upgraded.
Need to get 5,168 kB of archives.
After this operation, 25.6 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python3-numpy amd64 1:1.21.5-1ubuntu22.04.1 [3,467 kB]
Get:2 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy/main amd64 python3-gdal amd64 3.8.4+dfsg-1~jammy0 [1,095 kB]
Get:3 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy/main amd64 gdal-bin amd64 3.8.4+dfsg-1~jammy0 [605 kB]
Fetched 5,168 kB in 3s (1,539 kB/s)
Selecting previously unselected package python3-numpy.
(Reading database ... 118294 files and directori

In [18]:
# ══════════════════════════════════════════════════════════════════════
# Sample the raster at each settlement point.
#
# Note this is POINT SAMPLING, not a zonal statistic: elevation at the
# settlement itself, not an average over a buffer. Different question,
# different method.
# ══════════════════════════════════════════════════════════════════════
settlements = gpd.read_parquet(D + "settlements_sabah_full.parquet")

r      = rasterio.open("srtm_sabah.tif")
coords = [(g.x, g.y) for g in settlements.geometry]
settlements["elevation_m"] = [float(v[0]) for v in r.sample(coords)]

print(settlements["elevation_m"].describe().round(1).to_string())
print("\nhighest 5 — should be the Kinabalu foothills:")
print(settlements.nlargest(5, "elevation_m")[["name","place","elevation_m"]].to_string(index=False))

settlements.to_parquet(D + "settlements_sabah_full.parquet")
print("\nsaved ✓ D8 complete")

count    1448.0
mean      275.4
std       281.3
min        -2.0
25%        22.0
50%       215.0
75%       409.0
max      1510.0

highest 5 — should be the Kinabalu foothills:
               name   place  elevation_m
           Tinompok village       1510.0
    Kampung Mesilau  hamlet       1480.0
Kampung Kinasaraban  hamlet       1381.0
 Kampung Cinta Mata  hamlet       1327.0
   Kampung Sandatan village       1326.0

saved ✓ D8 complete


In [19]:
# ══════════════════════════════════════════════════════════════════════
# CHECKPOINT — verify the fused master file before building features.
# Every stage after this depends on it being correct.
# ══════════════════════════════════════════════════════════════════════
s = gpd.read_parquet(D + "settlements_sabah_full.parquet")
print(f"{len(s)} settlements")
print("columns:", sorted(s.columns.tolist()))
print("\nmissing values per column:")
print(s.isna().sum().to_string())
print("\n  name  missing = unnamed OSM places → ALWAYS join on settlement_id")
print("  rwi   missing = 'no wealth data'   → NOT 'poor'")
print("\ngeometry: 0 null required →", int(s.geometry.isna().sum()))
print("bounds (Sabah ≈ 115-120E, 4-7.5N):", s.total_bounds.round(2))

1448 settlements
columns: ['elevation_m', 'flood_prone', 'geometry', 'name', 'place', 'pop_2km', 'rwi', 'seasonal_water_px']

missing values per column:
name                 38
place                 0
geometry              0
pop_2km               0
seasonal_water_px     0
flood_prone           0
rwi                  96
elevation_m           0

  name  missing = unnamed OSM places → ALWAYS join on settlement_id
  rwi   missing = 'no wealth data'   → NOT 'poor'

geometry: 0 null required → 0
bounds (Sabah ≈ 115-120E, 4-7.5N): [115.36   4.23 119.21   7.35]


---
## P2 — Feature engineering: the settlement ↔ tile join

**Stage 2 → 3: information becomes features and indicators.**

This is the join the whole project depends on, and the place where a silent error would be
hardest to detect. Each settlement is buffered by 2 km and every Ookla tile intersecting
that buffer is attached to it.

### Method, and why each choice was made

| Step | Choice | Reason |
|---|---|---|
| Projection | EPSG:32650 before buffering | metres, not degrees — the trap from D4 |
| Buffer | 2 km around each settlement | matches `pop_2km`, so stakes and evidence describe the same zone |
| Predicate | `intersects`, not `within` | tiles are areas; a tile overlapping the buffer is relevant evidence |
| Aggregation | **median** across all matched tile-quarter rows | one quarter can be noisy or lucky; the median across four gives typical yearly experience and resists outliers |
| `n_tiles` | distinct physical tiles | a tile repeats across quarters; counting joined rows would inflate it |
| Missing speeds | left as `NaN` | never invented. No measurement means unknown, not slow. |

### Evidence tier

The tier is a statement about **how much we know**, kept strictly separate from how bad
things look:

- `measured` — at least 20 tests **and** at least 3 distinct tiles
- `insufficient` — fewer than 5 tests, or no matching tile at all
- `low_evidence` — everything in between

This separation is the project's central discipline: **absence of data is uncertainty, never
poor service.**

In [20]:
# ══════════════════════════════════════════════════════════════════════
# P2 — FEATURE JOIN
# Reads : settlements_sabah_full.parquet, tiles_sabah_2025q1..q4, facilities_sabah
# Writes: settlements_sabah_features_v1.parquet
#
# The master file is BACKED UP and never overwritten — the derived file is
# separate, so a bug here can be fixed by re-running rather than by
# rebuilding eight datasets.
# ══════════════════════════════════════════════════════════════════════
import shutil
UTM = 32650                       # projected CRS in metres, UTM zone 50N (Sabah)
BUF_TILES, BUF_FAC = 2000, 3000   # metres

shutil.copy(D + "settlements_sabah_full.parquet", D + "settlements_sabah_full_BACKUP.parquet")
print("backup written ✓")

# --- stable identifier -------------------------------------------------
# Names repeat and 38 are missing, so every downstream join uses this id.
s = gpd.read_parquet(D + "settlements_sabah_full.parquet").reset_index(drop=True)
s["settlement_id"] = ["S" + str(i).zfill(4) for i in range(len(s))]
assert s.settlement_id.is_unique
print(f"{len(s)} settlements, ids unique ✓")

# --- harmonise the four quarters into one table ------------------------
tiles = pd.concat([gpd.read_parquet(D + f"tiles_sabah_2025q{q}.parquet")
                   for q in tqdm([1,2,3,4], desc="quarters")], ignore_index=True)
tiles = gpd.GeoDataFrame(tiles, geometry="geometry", crs=4326)
print(f"{len(tiles)} tile-quarter rows | quarters: {sorted(tiles.quarter.unique())}")

# --- validate geometry BEFORE joining ----------------------------------
for nm, gdf in [("settlements", s), ("tiles", tiles)]:
    bad = (~gdf.geometry.is_valid).sum() + gdf.geometry.is_empty.sum() + gdf.geometry.isna().sum()
    print(f"{nm}: {bad} invalid/empty/null geometries, CRS={gdf.crs.to_epsg()}")
    assert bad == 0

# --- project to metres, buffer, spatial join ---------------------------
s_m     = s.to_crs(UTM)
tiles_m = tiles.to_crs(UTM).copy()
fac_m   = gpd.read_parquet(D + "facilities_sabah.parquet").to_crs(UTM)

buf = s_m[["settlement_id","geometry"]].copy()
buf["geometry"] = buf.buffer(BUF_TILES)

# A tile appears once per quarter; hashing its geometry gives a stable
# identity so `n_tiles` counts physical tiles, not joined rows.
tiles_m["tile_id"] = tiles_m.geometry.apply(lambda g: g.wkb_hex)

j = gpd.sjoin(tiles_m, buf, predicate="intersects").drop(columns="index_right")
print(f"{len(j)} matched tile-quarter ↔ settlement pairs")

# --- aggregate to one row per settlement -------------------------------
agg = j.groupby("settlement_id").agg(
    dl_mbps    = ("avg_d_kbps", lambda x: x.median()/1000),   # kbps → Mbps
    ul_mbps    = ("avg_u_kbps", lambda x: x.median()/1000),
    latency_ms = ("avg_lat_ms", "median"),
    n_tests    = ("tests", "sum"),
    n_tiles    = ("tile_id", "nunique"),
).reset_index()

s = s.merge(agg, on="settlement_id", how="left")
s["n_tests"] = s["n_tests"].fillna(0).astype(int)
s["n_tiles"] = s["n_tiles"].fillna(0).astype(int)
# speeds deliberately left NaN where nothing matched — never filled

# --- evidence tier -----------------------------------------------------
s["evidence_tier"] = np.where(
    (s.n_tests >= 20) & (s.n_tiles >= 3), "measured",
    np.where(s.n_tests < 5, "insufficient", "low_evidence"))

# --- anchor institutions within 3 km -----------------------------------
fbuf = s_m[["settlement_id","geometry"]].copy()
fbuf["geometry"] = fbuf.buffer(BUF_FAC)
fj     = gpd.sjoin(fac_m, fbuf, predicate="within")
counts = fj.groupby(["settlement_id","amenity"]).size().unstack(fill_value=0)
for c in ["school","clinic","hospital","doctors"]:
    if c not in counts: counts[c] = 0
s = s.merge(pd.DataFrame({
        "settlement_id": counts.index,
        "n_schools_3km": counts["school"].values,
        "n_clinics_3km": (counts["clinic"] + counts["hospital"] + counts["doctors"]).values,
    }), on="settlement_id", how="left")
s[["n_schools_3km","n_clinics_3km"]] = s[["n_schools_3km","n_clinics_3km"]].fillna(0).astype(int)

assert s.settlement_id.is_unique and s.geometry.notna().all()
s.to_parquet(D + "settlements_sabah_features_v1.parquet")
print("\nsaved ✓ settlements_sabah_features_v1.parquet")
print(sorted(s.columns.tolist()))

backup written ✓
1448 settlements, ids unique ✓


quarters:   0%|          | 0/4 [00:00<?, ?it/s]

20601 tile-quarter rows | quarters: ['2025q1', '2025q2', '2025q3', '2025q4']
settlements: 0 invalid/empty/null geometries, CRS=4326
tiles: 0 invalid/empty/null geometries, CRS=4326
37156 matched tile-quarter ↔ settlement pairs

saved ✓ settlements_sabah_features_v1.parquet
['dl_mbps', 'elevation_m', 'evidence_tier', 'flood_prone', 'geometry', 'latency_ms', 'n_clinics_3km', 'n_schools_3km', 'n_tests', 'n_tiles', 'name', 'place', 'pop_2km', 'rwi', 'seasonal_water_px', 'settlement_id', 'ul_mbps']


In [21]:
# ══════════════════════════════════════════════════════════════════════
# MANDATORY SANITY CHECKS — a join that runs without error can still be
# completely wrong. These four checks are run every time.
# ══════════════════════════════════════════════════════════════════════
print("(a) evidence tier distribution, %")
print((s.evidence_tier.value_counts(normalize=True)*100).round(1).to_string())

print("\n(b) overall median download:", round(s.dl_mbps.median(), 1), "Mbps")

m = s[s.evidence_tier == "measured"]
print("\n(c) FASTEST 10 — urban names expected")
print(m.nlargest(10,"dl_mbps")[["name","place","dl_mbps","n_tests","n_tiles"]].to_string(index=False))
print("\n(c) SLOWEST 10 — rural/interior names expected")
print(m.nsmallest(10,"dl_mbps")[["name","place","dl_mbps","n_tests","n_tiles"]].to_string(index=False))

print("\n(d) the five cities — all should be fast and well-measured")
print(s[s.place=="city"][["name","dl_mbps","n_tests","n_tiles","evidence_tier"]].to_string(index=False))

print("\nbounds:", s.geometry.total_bounds.round(2))

(a) evidence tier distribution, %
evidence_tier
measured        58.7
insufficient    23.1
low_evidence    18.2

(b) overall median download: 45.9 Mbps

(c) FASTEST 10 — urban names expected
                 name   place  dl_mbps  n_tests  n_tiles
       Kampung Melati  hamlet 328.6405       79       13
  Kampung Mostyn Lama village 312.6880      262       16
 Kampung Tagasan Tani  hamlet 303.7340      185       19
  Kampung Sri Bahagia village 291.6200      319       18
        Kampung Nipan  hamlet 247.8285      131       20
Kampung Berhala Darat village 244.3150      845       10
  Kampung Air Sim Sim village 244.3150     1086       10
      Kampung Kadazan village 238.5990      328       19
                Sugud    town 237.8620      894       31
       Park Residence village 232.0590     1073       39

(c) SLOWEST 10 — rural/interior names expected
             name   place  dl_mbps  n_tests  n_tiles
Kampung Nopungguk village   3.3810       20        4
           Nagaya village   5

### 📸 Visual check — the join itself

Run the cell below and **paste a screenshot of the map here.**

Red circle = the 2 km buffer. Blue rectangles = the Ookla tiles matched to it. The tiles must
sit inside or straddling the circle.

**Worth noting for the methodology page:** matched tiles often sit at the *edge* of the
buffer rather than on the settlement. That is inherent to the method, and it is why the
feature is described as the **nearby observed mobile environment**, not the speed at the
settlement.

In [22]:
# Visual verification of one settlement's join
import folium
sid  = s[s.evidence_tier=="measured"].sample(1, random_state=1).settlement_id.iloc[0]
row  = s[s.settlement_id == sid].iloc[0]
ring = buf[buf.settlement_id == sid].to_crs(4326)
mt   = j[j.settlement_id == sid].drop_duplicates("tile_id").to_crs(4326)

fmap = folium.Map(location=[row.geometry.y, row.geometry.x], zoom_start=13)
folium.GeoJson(ring, style_function=lambda x: {"color":"red","fill":False}).add_to(fmap)
folium.GeoJson(mt,   style_function=lambda x: {"color":"blue","weight":1}).add_to(fmap)
folium.Marker([row.geometry.y, row.geometry.x], tooltip=str(row["name"])).add_to(fmap)
print(f"{row['name']} — {len(mt)} distinct tiles, {row.n_tests} tests, {row.dl_mbps:.1f} Mbps")
fmap

Puralai — 3 distinct tiles, 37 tests, 136.7 Mbps


---
## DIPI v0 — Digital Inclusion Priority Index

**Stage 3: the indicator.**

DIPI is a **transparent weighted index**, not a machine-learning output. That is a deliberate
choice: there is no trusted label for "officially underserved", so a model would have nothing
honest to learn. A weighted score explains itself through its own components, which is what a
government planner can audit and argue with.

### Four pillars

| Pillar | Weight | Column | Direction |
|---|---|---|---|
| Connectivity deficit | 40% | `dl_mbps` | slower → higher priority |
| Population stakes | 25% | `pop_2km` | more people → higher |
| Anchor institutions | 15% | schools + clinics within 3 km | more → higher |
| Equity | 20% | `rwi` | poorer → higher |

Every pillar is converted to a **percentile rank** before weighting. One rule for all four,
immune to outliers (the fastest settlement is over 300 Mbps and would otherwise squash
everything else).

### Two deliberate exclusions

- **Flood context stays out of the score.** It is historic surface-water adjacency including
  tidal water, not a hazard model. It is available as a map lens instead.
- **Evidence confidence stays out of the score.** Blending it in would make "no data" look
  like "poor service".

### Two queues, not one list

Settlements with no usable evidence cannot be scored on connectivity, so they are **not
scored at all**. They form a separate queue ranked by stakes alone and labelled as needing
measurement. Missing RWI is treated as neutral (0.5), never as poor.

In [23]:
# ══════════════════════════════════════════════════════════════════════
# DIPI v0
# Reads : settlements_sabah_features_v1.parquet
# Writes: settlements_sabah_dipi_v1.parquet
# ══════════════════════════════════════════════════════════════════════
s = gpd.read_parquet(D + "settlements_sabah_features_v1.parquet")

WEIGHTS = {"connectivity": 0.40, "population": 0.25, "institutions": 0.15, "equity": 0.20}
assert abs(sum(WEIGHTS.values()) - 1) < 1e-9

# --- two queues --------------------------------------------------------
scored = s[s.evidence_tier.isin(["measured","low_evidence"])].copy()   # Queue A
gap    = s[s.evidence_tier == "insufficient"].copy()                   # Queue B
print(f"Queue A (scored): {len(scored)}   Queue B (evidence gap): {len(gap)}")

# --- pillars, each as a percentile rank --------------------------------
scored["p_connectivity"] = (-scored.dl_mbps).rank(pct=True)            # negate: slow ranks high
scored["p_population"]   = np.log1p(scored.pop_2km).rank(pct=True)     # log: 50k vs 100k matters
scored["p_institutions"] = (scored.n_schools_3km + scored.n_clinics_3km).rank(pct=True)
scored["p_equity"]       = (-scored.rwi).rank(pct=True).fillna(0.5)    # missing → NEUTRAL

scored["dipi"] = ((WEIGHTS["connectivity"] * scored.p_connectivity +
                   WEIGHTS["population"]   * scored.p_population   +
                   WEIGHTS["institutions"] * scored.p_institutions +
                   WEIGHTS["equity"]       * scored.p_equity) * 100).round(1)
scored["rank"] = scored.dipi.rank(ascending=False, method="min").astype(int)

# --- Queue B: stakes only, explicitly NOT a DIPI score -----------------
gap["stakes_score"] = ((np.log1p(gap.pop_2km).rank(pct=True) * 0.6 +
                        (gap.n_schools_3km + gap.n_clinics_3km).rank(pct=True) * 0.4) * 100).round(1)
gap["gap_rank"] = gap.stakes_score.rank(ascending=False, method="min").astype(int)
for c in ["dipi","rank","p_connectivity","p_population","p_institutions","p_equity"]:
    gap[c] = np.nan          # never scored — the whole point

out = gpd.GeoDataFrame(pd.concat([scored, gap], ignore_index=True),
                       geometry="geometry", crs=s.crs)
out.to_parquet(D + "settlements_sabah_dipi_v1.parquet")
print("saved ✓ settlements_sabah_dipi_v1.parquet")

Queue A (scored): 1114   Queue B (evidence gap): 334
saved ✓ settlements_sabah_dipi_v1.parquet


In [24]:
# ══════════════════════════════════════════════════════════════════════
# DIPI VALIDATION — an index cannot be validated against ground truth,
# so it is validated against expectations that must hold if it is sane.
# ══════════════════════════════════════════════════════════════════════
print("DIPI distribution:"); print(scored.dipi.describe().round(1).to_string())

print("\nTOP 15 PRIORITY (Queue A)")
print(scored.nsmallest(15,"rank")[["rank","name","place","dl_mbps","pop_2km",
      "n_schools_3km","rwi","dipi"]].to_string(index=False))

print("\nCITIES — must rank LOW priority (they are fast)")
print(scored[scored.place=="city"][["name","dl_mbps","dipi","rank"]].to_string(index=False))

print("\nTOP 10 EVIDENCE GAP (Queue B — unscored, needs measurement)")
print(gap.nsmallest(10,"gap_rank")[["gap_rank","name","place","pop_2km","stakes_score"]].to_string(index=False))

print("\nTop-50 population — the index must not surface empty hamlets:")
print(scored.nsmallest(50,"rank").pop_2km.describe().round(0).to_string())

print("\nPILLAR CORRELATIONS — want moderate. Near 1.0 would mean a redundant pillar.")
print(scored[["p_connectivity","p_population","p_institutions","p_equity"]].corr().round(2).to_string())

DIPI distribution:
count    1114.0
mean       50.0
std        11.2
min        20.2
25%        41.6
50%        50.2
75%        58.8
max        76.6

TOP 15 PRIORITY (Queue A)
 rank                    name   place  dl_mbps  pop_2km  n_schools_3km       rwi  dipi
    1         Kampung Tangkol village  16.4510     1651              1 -0.352250  76.6
    2                   Talas village  11.3800     1538              2 -0.143667  75.9
    2        Kampung Gentuong village   8.4635     1436              1 -0.196500  75.9
    4              Salimandut village  19.5770     1744              1 -0.324333  75.4
    5       Sindungon Panjang village   8.7180     1411              0 -0.148000  74.8
    6           Kampung Rosok village  19.5970     1160              2 -0.279500  74.5
    7             Tanah Merah village  21.9100     5397              4  0.053333  74.2
    8 Kampung Pongoputan Baru village  19.1300     1297              1 -0.313667  74.1
    9       Kampung Mengkulat village  15.9

In [25]:
# ══════════════════════════════════════════════════════════════════════
# WEIGHT SENSITIVITY — the weights are a judgement, so the honest question
# is how much the ranking depends on them. Three alternative weightings
# are compared against the shipped one by top-50 overlap.
# ══════════════════════════════════════════════════════════════════════
top50_base = set(scored.nsmallest(50, "rank").settlement_id)

for w in [{"c":.30,"p":.30,"i":.15,"e":.25},
          {"c":.50,"p":.20,"i":.15,"e":.15},
          {"c":.40,"p":.25,"i":.25,"e":.10}]:
    alt = (w["c"]*scored.p_connectivity + w["p"]*scored.p_population +
           w["i"]*scored.p_institutions + w["e"]*scored.p_equity)
    r   = alt.rank(ascending=False, method="min")
    ov  = len(top50_base & set(scored.assign(r=r).nsmallest(50,"r").settlement_id))
    print(f"{w} → top-50 overlap: {ov}/50")

print("\nHigh overlap = the ranking is driven by the data, not by our weights.")

{'c': 0.3, 'p': 0.3, 'i': 0.15, 'e': 0.25} → top-50 overlap: 44/50
{'c': 0.5, 'p': 0.2, 'i': 0.15, 'e': 0.15} → top-50 overlap: 44/50
{'c': 0.4, 'p': 0.25, 'i': 0.25, 'e': 0.1} → top-50 overlap: 36/50

High overlap = the ranking is driven by the data, not by our weights.


In [26]:
# ══════════════════════════════════════════════════════════════════════
# EXPORT for the dashboard — GeoJSON, the format web maps read directly.
# ══════════════════════════════════════════════════════════════════════
dipi = gpd.read_parquet(D + "settlements_sabah_dipi_v1.parquet")
dipi.to_file(D + "dipi.geojson", driver="GeoJSON")

districts = gpd.read_file(D + "gadm41_MYS_2.json")
districts[districts.NAME_1 == "Sabah"].to_file(D + "sabah_districts.geojson", driver="GeoJSON")
print("exported ✓ dipi.geojson + sabah_districts.geojson")

exported ✓ dipi.geojson + sabah_districts.geojson


---
## OpenCelliD — telecom supply context layer

Crowdsourced cell-tower records, used as a **context layer only**. It is tested as a model
feature in notebook 02 and rejected there; this cell just prepares the data so that test can
be run.

⚠️ In this schema **`lon` comes before `lat`**. Swapping them puts every record in the wrong
hemisphere.

Licence: CC BY-SA 4.0. Visible credit to *OpenCelliD* with a link to opencellid.org is
required wherever this appears.

The download endpoint allows only two requests per token per day, so a cached export
(`502.csv.gz`) is read from Drive.

In [29]:
# ══════════════════════════════════════════════════════════════════════
# OpenCelliD — load the Malaysia export (MCC 502) and clip to Sabah.
# The gzip magic-byte check matters: when the endpoint rate-limits, it
# returns a JSON error with a .csv.gz filename, which would otherwise be
# read as data.
# ══════════════════════════════════════════════════════════════════════
RAW = D + "502.csv.gz"
assert os.path.exists(RAW), f"not found: {RAW} — ask the team for the OpenCelliD export"
assert open(RAW,"rb").read(2) == b"\x1f\x8b", "not a gzip file — likely a rate-limit error page"
print(f"valid gzip, {os.path.getsize(RAW)/1e6:.2f} MB ✓")

COLS = ["radio","mcc","net","area","cell","unit","lon","lat","range",
        "samples","changeable","created","updated","averageSignal"]
ocid = pd.read_csv(RAW, names=COLS, header=None, compression="gzip")
print(f"{len(ocid):,} Malaysia cell records | mcc: {ocid.mcc.unique()[:3]}")

sab = ocid[(ocid.lon.between(115.0, 119.7)) & (ocid.lat.between(4.0, 7.6))].copy()
sab.to_parquet(D + "ocid_sabah.parquet")
print(f"{len(sab):,} inside the Sabah bounding box → ocid_sabah.parquet ✓\n")

print(sab.radio.value_counts().to_string())
print(f"\ndistinct operators: {sab.net.nunique()}")
print(f"updated range: {pd.to_datetime(sab.updated, unit='s').min().date()} → "
      f"{pd.to_datetime(sab.updated, unit='s').max().date()}")

valid gzip, 1.86 MB ✓
95,305 Malaysia cell records | mcc: [502]
1,217 inside the Sabah bounding box → ocid_sabah.parquet ✓

radio
LTE    1171
GSM      46

distinct operators: 7
updated range: 2025-02-19 → 2026-08-02


---
## Notebook 1 complete

### Files now in `MyDrive/Dataset/`

| File | Contents |
|---|---|
| `settlements_sabah_full.parquet` | the fused master — D3 + D4 + D6 + D7 + D8 |
| `settlements_sabah_full_BACKUP.parquet` | safety copy taken before P2 |
| `settlements_sabah_features_v1.parquet` | master + Ookla speeds + evidence tier + facility counts |
| `settlements_sabah_dipi_v1.parquet` | features + DIPI + pillar scores + two queues |
| `dipi.geojson`, `sabah_districts.geojson` | dashboard inputs |
| `facilities_sabah.parquet` | 606 mapped schools and clinics |
| `tiles_sabah_2025q1…q4.parquet` | clipped Ookla quarters |
| `ocid_sabah.parquet` | OpenCelliD context layer |
| `gadm41_MYS_1.json`, `gadm41_MYS_2.json` | boundaries |

Intermediate files `settlements_sabah.parquet` and `settlements_sabah_pop.parquet` are
build artefacts and are superseded by the master.

### After this notebook
Run [`export_training_table.py`](https://github.com/RextonRZ/dino-slayer/blob/main/dataset/export_training_table.py)
→ `training_table.csv`. Copy it into `MyDrive/Dataset/`.


### Credits

Network performance © Ookla Speedtest Open Data (CC BY-NC-SA 4.0) · © OpenStreetMap
contributors (ODbL) · Population © WorldPop (CC BY 4.0) · Surface water © EC JRC/Google ·
Relative Wealth Index © Meta Data for Good (CC BY 4.0) · Elevation: NASA SRTM ·
Boundaries: GADM · Cell records © OpenCelliD (CC BY-SA 4.0)